# Step 2: Clean the data into three tables

**Goal:** turn raw rows into
- **purchases**: real sales (counts: Frequency, AOV, ...)
- **returns**: cancellations, with negative `TotalPrice` (return-rate feature)
- **ledger**: purchases + returns stacked, i.e. signed money (Monetary and the 6-month target = money *kept*)

## Decisions taken
1. **Net returns off spend.** A fully cancelled order is worth 0, not its gross value. Returns are stacked as negative rows dated at the cancellation, so spend in any window = sum of that window's rows.
2. **Drop internal stock codes** (manual entries, adjustments, bank charges, tests). **Keep postage/carriage** (`POST`, `C2`, `DOT`): customers really pay it.

## Rules (in order; the logic lives in `src/step2_clean.py`)
1. Drop rows with no Customer ID.
2. Drop internal stock codes.
3. Invoices starting with `C` are cancellations and go to the returns table.
4. From the rest, drop `Quantity <= 0`.
5. From the rest, drop `Price <= 0`.
6. Add `TotalPrice = Quantity * Price`.

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from step1_load_explore import load_raw
from step2_clean import clean, build_ledger, INTERNAL_CODES

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

## 1. Run the cleaning and check the row accounting

Every raw row must end up in exactly one place. `clean()` asserts that the pieces add up to the raw total.

In [2]:
raw = load_raw()
purchases, returns, acct = clean(raw)
ledger = build_ledger(purchases, returns)
pd.Series(acct, name="rows").to_frame()

,rows
raw rows,1067371
dropped: no Customer ID,243007
dropped: internal stock codes,1428
moved to returns (C invoices),18136
dropped: Quantity <= 0,0
dropped: Price <= 0,61
purchases kept,804739


`dropped: Quantity <= 0` is **0**: every negative quantity was already on a `C` invoice, so the cancellation rule caught them. The negatives we saw in step 1 were really cancellations.

In [3]:
# Proof: no negative quantity exists outside cancelled invoices
not_cancel = ~raw["Invoice"].astype(str).str.startswith("C")
print("Negative-quantity rows on non-C invoices (with a customer):",
      ((raw["Quantity"] <= 0) & not_cancel & raw["Customer ID"].notna()).sum())

Negative-quantity rows on non-C invoices (with a customer): 0


## 2. Decision 2: internal stock codes

Real products have codes starting with 5 digits. Here is what we dropped, and what non-product codes remain.

In [4]:
dropped = raw[raw["Customer ID"].notna() & raw["StockCode"].astype(str).isin(INTERNAL_CODES)].copy()
dropped["Value"] = dropped["Quantity"] * dropped["Price"]
print("DROPPED internal codes:")
display(dropped.groupby("StockCode")["Value"].agg(["count", "sum"]).sort_values("count", ascending=False))

odd = purchases[~purchases["StockCode"].astype(str).str.match(r"^\d{5}")]
print("KEPT non-product codes in purchases (postage / carriage):")
display(odd.groupby("StockCode")["TotalPrice"].agg(["count", "sum"]).sort_values("sum", ascending=False))

DROPPED internal codes:


,count,sum
StockCode,,
M,1115,-185153.230
D,174,-13390.250
ADJUST,61,2038.120
BANK CHARGES,38,345.001
PADS,19,-36.583
TEST001,15,202.500
ADJUST2,3,731.050
SP1002,2,14.750
TEST002,1,1.000


KEPT non-product codes in purchases (postage / carriage):


,count,sum
StockCode,,
POST,1838,126563.04
C2,253,12796.00
DOT,16,11906.36


## 3. The purchases table

In [5]:
print("customers:", purchases["Customer ID"].nunique(), "| invoices:", purchases["Invoice"].nunique())
print("dates:", purchases["InvoiceDate"].min(), "->", purchases["InvoiceDate"].max())
display(purchases.head())
purchases[["Quantity", "Price", "TotalPrice"]].describe()

customers: 5854 | invoices: 36711
dates: 2009-12-01 07:45:00 -> 2011-12-09 12:50:00


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalPrice
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0


,Quantity,Price,TotalPrice
count,804739.000000,804739.000000,804739.000000
mean,13.291301,3.021709,21.852694
std,143.681580,11.066490,222.529141
min,1.000000,0.030000,0.060000
25%,2.000000,1.250000,4.950000
50%,5.000000,1.950000,11.850000
75%,12.000000,3.750000,19.500000
max,80995.000000,8142.750000,168469.600000


## 4. The returns table

`TotalPrice` is negative (money handed back). Some returns belong to customers with no purchases in our data; they are ignored when we build per-customer features.

In [6]:
print("customers:", returns["Customer ID"].nunique(), "| invoices:", returns["Invoice"].nunique())
print("return rows whose customer never purchased:", (~returns["Customer ID"].isin(purchases["Customer ID"])).sum())
print("customers with both purchases and returns:", len(set(returns["Customer ID"]) & set(purchases["Customer ID"])))
returns.head()

customers: 2465 | invoices: 7415
return rows whose customer never purchased: 88


customers with both purchases and returns: 2442


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalPrice
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321,Australia,-35.4
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321,Australia,-9.9
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321,Australia,-17.0
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321,Australia,-12.6
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321,Australia,-35.4


## 5. Decision 1: netting returns (the ledger)

Compare gross spend with net spend. Earlier we saw that the 4 biggest orders were each fully cancelled.

In [7]:
print(f"gross purchases: {purchases['TotalPrice'].sum():>14,.0f}")
print(f"returns:         {returns['TotalPrice'].sum():>14,.0f}")
print(f"net (ledger):    {ledger['TotalPrice'].sum():>14,.0f}")

gross_c = purchases.groupby("Customer ID")["TotalPrice"].sum().rename("gross")
net_c = ledger.groupby("Customer ID")["TotalPrice"].sum().rename("net")
cmp = pd.concat([gross_c, net_c], axis=1).fillna(0)
print("customers with NEGATIVE net spend overall:", (cmp["net"] < 0).sum())

gross purchases:     17,585,715
returns:               -742,175
net (ledger):        16,843,540


customers with NEGATIVE net spend overall: 28


The top customers ranked by **gross** vs **net** spend. Big gaps are cancelled orders that gross spend would have wrongly counted as value.

In [8]:
cmp["cancelled"] = cmp["gross"] - cmp["net"]
print("Top 8 customers by GROSS spend:")
display(cmp.nlargest(8, "gross").round(0))
print("Top 8 customers by NET spend:")
display(cmp.nlargest(8, "net").round(0))

Top 8 customers by GROSS spend:


,gross,net,cancelled
Customer ID,,,
18102,608822.0,606243.0,2578.0
14646,528603.0,524829.0,3774.0
14156,305229.0,299926.0,5302.0
14911,284030.0,270163.0,13867.0
17450,246973.0,234894.0,12079.0
13694,196483.0,191196.0,5287.0
17511,175604.0,171897.0,3707.0
16446,168472.0,3.0,168470.0


Top 8 customers by NET spend:


,gross,net,cancelled
Customer ID,,,
18102,608822.0,606243.0,2578.0
14646,528603.0,524829.0,3774.0
14156,305229.0,299926.0,5302.0
14911,284030.0,270163.0,13867.0
17450,246973.0,234894.0,12079.0
13694,196483.0,191196.0,5287.0
17511,175604.0,171897.0,3707.0
12415,144383.0,143194.0,1189.0


**Caveat we will revisit in step 5:** timing. If someone buys in the feature window and cancels in the target window, their target spend can go negative. We keep it honest for now and will count how often it happens before deciding anything.

## 6. Save

`python src/step2_clean.py` writes `data/purchases.csv`, `data/returns.csv` and `data/ledger.csv`. Later steps read those.

## What we learned

- 804,739 purchase rows (5,854 customers); 18,136 return rows (2,465 customers).
- All negative quantities were cancellations.
- Netting removes about £742k (4%) from £17.6M gross spend. Overall the effect is small, but for a few customers it is huge: customer 16446 goes from £168k gross to about £3 net, so gross spend would have ranked them #8 by value.
- Features that count things (Frequency, AOV) use `purchases`; money amounts (Monetary, target) use `ledger`.

**Next (step 3):** the time split: feature window vs target window.